# Chessboard Dataset Generation (streaming, HF push)

Generates Task 1 / Task 2 / Task 3 samples from the Lichess puzzle database and pushes each task/split straight to the Hugging Face Hub as a Parquet-backed dataset, rendering and encoding one image at a time instead of materializing the whole dataset on disk or in memory.

Companion to `Datasets_creator.ipynb` (disk-folder based); this notebook uses `datasets.Dataset.from_generator` + `push_to_hub` instead, so the resulting datasets also support `load_dataset(..., streaming=True)` on read.

In [ ]:
!pip install -q python-chess cairosvg datasets huggingface_hub scikit-learn tqdm


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

import pandas as pd


In [ ]:
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}


## Clone the repo (so we can import `src/data`'s generation & upload functions)

In [ ]:
if CONFIG["colab"]:
    repo_dir = Path(CONFIG["repo_dir"])
    if repo_dir.exists():
        subprocess.run(["rm", "-rf", str(repo_dir)], check=True)

    repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_dir)],
        capture_output=True, text=True
    )
    assert result.returncode == 0, f"Git clone failed: {result.stderr}"

    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path(".").resolve()
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))

sys.path.insert(0, str(repo_dir / "src" / "data"))

REPO_ROOT = Path(".").resolve()
print("Setup complete. REPO_ROOT:", REPO_ROOT)


In [ ]:
from src.data.utilities import (
    load_lichess_csv,
    balanced_turn_split,
    build_hf_dataset,
    push_task_dataset_to_hub,
    authenticate_hf,
)


## Authenticate with Hugging Face

Add your token once as a Colab secret (key icon in the left sidebar) named `HF_TOKEN` (needs **write** access to the `bdatm-project` org), and toggle "Notebook access" on for it. This cell logs in non-interactively so the later push cells don't stall waiting on an input prompt.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

assert hf_token, (
    "No HF token found. Add one as a Colab secret named 'HF_TOKEN' "
    "(key icon in the left sidebar), or set os.environ['HF_TOKEN'] manually."
)

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")


## Load the Lichess puzzle database

In [ ]:
csv_path = Path("lichess_db_puzzle.csv")
zst_path = Path("lichess_db_puzzle.csv.zst")

if not csv_path.exists():
    if shutil.which("zstd") is None and shutil.which("unzstd") is None:
        print("Installing zstd decompression tool...")
        !apt-get update -qq && apt-get install -y zstd -qq

    if not zst_path.exists():
        print("Downloading Lichess puzzles database (~250 MB compressed)...")
        !wget -q --show-progress https://database.lichess.org/lichess_db_puzzle.csv.zst

    print("Decompressing archive (~1.5 GB CSV)...")
    !unzstd -f --rm lichess_db_puzzle.csv.zst -o lichess_db_puzzle.csv
    !rm -f lichess_db_puzzle.csv.zst.*
    print("Decompression complete!")
else:
    print(f"'{csv_path}' already exists.")


## Balance by side-to-move and split 80/10/10

In [ ]:
# Total puzzles pulled from the CSV before balancing; N_PER_TURN*2 must be <= this.
MAX_SAMPLES = 20000
N_PER_TURN = 2000  # -> 2000 white-to-move + 2000 black-to-move = 4000 puzzles total

df_raw = load_lichess_csv(csv_path=csv_path, max_samples=MAX_SAMPLES)
splits = balanced_turn_split(df_raw, n_per_turn=N_PER_TURN, seed=42)

for split_name, split_df in splits.items():
    print(split_name, len(split_df), dict(split_df["turn"].value_counts()))


## Generate + push to Hugging Face (streamed, low memory)

For each task/split, `build_hf_dataset` renders and encodes samples one at a time via `Dataset.from_generator` (no full-dataset materialization in memory), and `push_task_dataset_to_hub` uploads the resulting Parquet-backed dataset straight to the Hub.

In [ ]:
ORGANIZATION_NAME = "bdatm-project"
TASKS = ["task1", "task2", "task3"]
IMAGE_SIZE = 512
PRIVATE = False

for task in TASKS:
    dataset_name = f"chess-{task}"
    print(f"\n==================== {task.upper()} ====================")
    for split_name, split_df in splits.items():
        push_task_dataset_to_hub(
            df=split_df,
            task=task,
            dataset_name=dataset_name,
            split=split_name,
            namespace=ORGANIZATION_NAME,
            image_size=IMAGE_SIZE,
            private=PRIVATE,
        )

print("\nAll task datasets pushed to the Hub.")


## Verify: stream a pushed dataset back without downloading it fully

In [ ]:
from datasets import load_dataset

verify_task = "task1"
repo_id = f"{ORGANIZATION_NAME}/chess-{verify_task}"

streamed = load_dataset(repo_id, split="train", streaming=True)
example = next(iter(streamed))

print({k: v for k, v in example.items() if k != "image"})
example["image"]
